# Book generation -- End to end
- Generate the books and the corresponding question/answer pairs
- If the data already exists, only load books and regenerate QA pairs
- Duration for loading the books and regenerate the questions: 1 minute

In [ ]:
from pathlib import Path
git_repo_filepath = str(Path.cwd().resolve().parents[2])  # repository root (this notebook: src/In_vitro_experiments/experiments)
import sys
import os

# Add the project root to sys.path
if str(Path(git_repo_filepath) / 'src') not in sys.path:
    sys.path.insert(0, str(Path(git_repo_filepath) / 'src')) # Insert at the beginning to prioritize it

print(f"Project root added to sys.path: {git_repo_filepath}")
print(f"Current sys.path: {sys.path}")
os.environ["CUDA_VISIBLE_DEVICES"] = "" 
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
from In_vitro_experiments.generation.benchmark_generation_wrapper import BenchmarkGenerationWrapper
book_parameters = {'indexing': 'default', 'nb_summaries': 0}
data_folder = Path(git_repo_filepath) / 'data' / 'narratives'
env_file = Path(git_repo_filepath) / '.env'

print("Generation with Claude -- 10 events")
prompt_parameters = {'nb_events': 10, 'name_universe': 'default', 'name_styles': 'concise', 'seed': 99, 'distribution_events': {'name': 'geometric', 'param': 0.3}}
model_parameters = {'model_name': 'claude-3-5-sonnet-20240620', 'max_new_tokens': 4096, 'itermax': 10} # itermax is integer, 1 for a single try
benchmark_claude_10 = BenchmarkGenerationWrapper(prompt_parameters, model_parameters, book_parameters, data_folder, env_file,rechecking=False)
my_benchmark = benchmark_claude_10
max_chapter = 2
print("Little book generated")

In [ ]:
print(my_benchmark.nb_chapters())

In [ ]:
from In_vitro_experiments.evaluation.generator_answers_1_prompting import generate_answers_func
evaluation_parameters = {
    'kind': 'prompting',
    'model_name': 'claude-3-5-sonnet-20240620',  # This is the correct stable model name
    'max_new_tokens': 2048  
}
df_generated_answers = generate_answers_func(my_benchmark, answering_parameters, data_folder, env_file)

# Exploration

In [ ]:
# Selection of the book of interest
my_benchmark = benchmark_claude_10

#### Initial exploration

In [ ]:
# Get event 0, as a list
my_benchmark.events[0]

In [ ]:
from In_vitro_experiments.generation.prompts import *
user_prompt = generating_question_chapter_prompt(my_benchmark.get_chapter(0))
response = my_benchmark.llm_questions_generated(user_prompt)
response


In [ ]:
# Get metadata for event 0, as a dictionary
my_benchmark.meta_events[0]

#### Raw generated sample indexed by (event, iteration), before chaptering [debug]

In [ ]:
# Print the successful iteration for an event_idx (those are raw samples, it is not the chapter indexing yet)
event_idx = 3
my_benchmark.pretty_print_debug_event_idx(event_idx)

In [ ]:
# Print the `event_idx` that are all invalid, even after all the iterations
invalid_samples = my_benchmark.invalid_debug_event_idx_func()
print(f"Invalid sample(s): {invalid_samples}")

In [ ]:
# Print details for an invalid sample
if len(invalid_samples) > 0:
    my_benchmark.pretty_print_debug_event_iter_idx(invalid_samples[0]) # by default take the last iteration

In [ ]:
# Print a specific event_idx and iter_idx (for debugging)
my_benchmark.pretty_print_debug_event_iter_idx(3,0)

#### At the book side

In [ ]:
# Get the number of tokens
print(f"{my_benchmark.nb_tokens()} tokens in the book")
# Get the number of chapters
print(f"{my_benchmark.nb_chapters()} chapters in the book")

#### Ground truth

In [ ]:
# Ground truth for each chapter
df_book_groundtruth = my_benchmark.df_book_groundtruth
df_book_groundtruth

#### Questions

In [ ]:
# Selected questions
df_qa = my_benchmark.df_qa
# df_qa.iloc[0][['question', 'correct_answer', 'correct_answer_chapters']]
df_qa

In [ ]:
# Widespreadness of the questions (with the default bins)
my_benchmark.df_qa_debug_widespreadness

In [ ]:
# Complete list of questions related to a single chapter, used for fine-tuning
my_benchmark.finetuning_questions_one_chapter